# Практичне завдання №1

## Requirements & Estimation Readiness Agent

**Авторка:** Anna Malkova
**Модель:** `gemini-3.1-flash-lite`
**Стек:** LangGraph, LangChain, ChromaDB,
Pydantic v2, SQLite
**Архітектури:** ReAct та Plan-and-Execute

## 1. Мета та бізнес-контекст

Агент підтримує перехід від discovery до delivery
estimation:

`Demand → Discovery → Requirements Readiness
→ Estimation → Delivery`

Основні функції:

- перевірка requirements readiness;
- класифікація estimation complexity;
- пошук handover gaps;
- Agentic RAG;
- SQLite persistence;
- HITL перед ризиковою відправкою;
- JSON trajectory logging.

## 2. Acceptance criteria

| Критерій | Реалізація |
|---|---|
| Domain tools | 4 Pydantic tools |
| ReAct | LLM–tools–LLM LangGraph |
| Safety | steps, timeout, repeated calls |
| Plan-and-Execute | planner/executor/replanner |
| Persistence | SqliteSaver та thread_id |
| RAG | 12 документів у ChromaDB |
| HITL | approve/reject/edit |
| Tests | 59 unit/integration tests |
| Trajectory | обидві архітектури |
| Bonus | comparison, visualization, fallback |

## 3. ReAct architecture

            ```mermaid
            ---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> finalize;
	agent -.-> tools;
	tools --> agent;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

            ```

            ReAct самостійно обирає tools і продовжує цикл
            reasoning після кожного observation.

In [ ]:
from safety import SafetyController
from react_agent import (
    REACT_TOOLS,
    build_react_graph,
    run_react_agent,
)

react_graph = build_react_graph(
    SafetyController()
)

print(list(react_graph.get_graph().nodes))
print([tool.name for tool in REACT_TOOLS])

### ReAct safety

- `max_steps = 10`;
- `timeout = 120` секунд;
- максимум два однакові tool calls;
- structured final response;
- статус `safety_stop`.

In [ ]:
RUN_LIVE = False

request = (
    "Для DEM-050 перевір requirements readiness, "
    "визнач estimation complexity та знайди "
    "правила у knowledge base."
)

if RUN_LIVE:
    response = run_react_agent(request)
    print(response)
else:
    print(
        "Live call вимкнений. "
        "Результат є у trajectory.json."
    )

## 4. Plan-and-Execute architecture

            ```mermaid
            ---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	executor(executor)
	approval(approval)
	replanner(replanner)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	approval --> replanner;
	executor -.-> approval;
	executor -.-> replanner;
	planner --> executor;
	replanner -. &nbsp;end&nbsp; .-> __end__;
	replanner -.-> executor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

            ```

            Planner формує structured plan. Executor виконує
            кожен step через вкладений ReAct. Replanner обирає
            `continue`, `replan` або `finish`. Approval node
            зупиняє risky operation до рішення людини.

In [ ]:
from plan_execute import (
    build_graph,
    create_initial_state,
)

plan_graph = build_graph()

print(list(plan_graph.get_graph().nodes))

create_initial_state(
    "Перевір readiness DEM-050."
)

## 5. Domain tools

1. `check_requirements_readiness`
2. `classify_estimation_complexity`
3. `identify_handover_gaps`
4. `submit_estimation_request`

Окремий RAG tool:

5. `search_delivery_knowledge`

Усі domain tools мають Pydantic v2 schemas,
validators та стандартну JSON-відповідь
`{status, data, error}`.

In [ ]:
from tools import DOMAIN_TOOLS

for tool in DOMAIN_TOOLS:
    print(
        tool.name,
        tool.args_schema.__name__,
    )

## 6. Agentic RAG

Persistent ChromaDB collection містить 12
документів про Definition of Ready, FR/NFR,
acceptance criteria, integrations, data migration,
security, dependencies, sizing та handover.

Агент сам вирішує, коли викликати semantic search.

In [ ]:
from knowledge import (
    initialize_knowledge_base,
    search_delivery_knowledge,
)

print(initialize_knowledge_base())

result = search_delivery_knowledge.invoke(
    {
        "query": (
            "requirements readiness "
            "before estimation"
        )
    }
)

print(result)

## 7. Human-in-the-Loop

`submit_estimation_request` є high-risk operation.

Перед виконанням людина бачить tool та всі
arguments і може обрати:

- `approve`;
- `reject`;
- `edit`.

Live demo:

- DEM-060 схвалено, створено `EST-001`;
- DEM-061 відхилено;
- для DEM-061 tool не виконувався.

In [ ]:
import json
from pathlib import Path

submission_path = Path(
    "submitted_estimation_requests.json"
)

if submission_path.exists():
    submissions = json.loads(
        submission_path.read_text(
            encoding="utf-8"
        )
    )

    print(
        [
            (
                item["request_id"],
                item["initiative_id"],
                item["status"],
            )
            for item in submissions
        ]
    )

## 8. SQLite persistence

LangGraph state зберігається через `SqliteSaver`
у `agent_state.db`.

Live demo показав:

- окремі checkpoints для двох thread_id;
- відновлення з `executor`, а не з початку;
- незалежний progress;
- завершення після кількох restart/resume.

In [ ]:
from persistence_demo import (
    compare_threads,
    inspect_thread,
)

completed = inspect_thread(
    "practice-persistence-002"
)

print(
    completed["state"]["status"],
    completed["state"]["current_step"],
    completed["state"]["used_tools"],
)

## 9. Trajectory logging

Кількість збережених runs: **2**.

`trajectory.json` містить ReAct та
Plan-and-Execute runs із messages, tools,
final response, safety та metadata.

In [ ]:
trajectory = json.loads(
    Path("trajectory.json").read_text(
        encoding="utf-8"
    )
)

for run in trajectory["runs"]:
    print(
        run["agent_type"],
        len(run["messages"]),
        run["final_response"].get(
            "used_tools",
            [],
        ),
    )

## 10. Числове порівняння

| Метрика | ReAct | Plan-and-Execute |
|---|---:|---:|
| Execution time | 6.286 s | 17.793 s |
| Quality score | 100.0% | 100.0% |
| Tool coverage | 100% | 100% |

ReAct був швидшим для короткого сценарію.
Plan-and-Execute надав явний plan, persistence,
replanning та HITL.

In [ ]:
comparison = json.loads(
    Path("comparison_results.json").read_text(
        encoding="utf-8"
    )
)

comparison["comparison"]

## 11. Автоматичні тести

Тести покривають schemas, tools, safety, RAG,
ReAct, Plan-and-Execute, replanning, HITL,
persistence, thread independence та comparison.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    capture_output=True,
    text=True,
    check=False,
)

print(result.stdout)
assert result.returncode == 0

## 12. Висновки

ReAct доцільний для коротких динамічних задач із
мінімальною latency.

Plan-and-Execute краще підходить для довгих
workflow, де потрібні планування, progress tracking,
persistence, replanning, HITL та auditability.

### Обмеження

- локальний mock storage замість Jira;
- навчальна knowledge base;
- залежність від Gemini quota;
- SQLite не є distributed production storage;
- quality score не замінює LLM-as-a-judge.

## 13. Основні артефакти

- `README.md`
- `trajectory.json`
- `agent_state.db`
- `comparison_results.json`
- `test_results.json`
- `react_graph.mmd`
- `plan_execute_graph.mmd`
- `Task_001_Malkova_Requirements_Estimation.ipynb`